# Road Damage Image Test

In [2]:
from ultralytics import YOLO
import io
from PIL import Image
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

model = YOLO(r"D:\Programs\Python\Road_Damage\RDD2022-India.v6-rdd2022-india-aug.yolov11\road_damage_yolo11n.pt")

names = [
    "D00_Longitudinal_Crack",
    "D10_Transverse_Crack",
    "D20_Alligator_Crack",
    "D40_Pothole"
]

upload = widgets.FileUpload(
    accept="image/*",
    multiple=False,
    description="Upload / Drop Image"
)

output = widgets.Output()

def predict_image(change):
    with output:
        clear_output()

        if not upload.value:
            return

        file_info = upload.value[0]
        image = Image.open(io.BytesIO(file_info["content"])).convert("RGB")

        results = model.predict(
            source=image,
            conf=0.25,
            imgsz=640,
            device=0,
            verbose=False
        )

        result_img = results[0].plot()

        plt.figure(figsize=(12, 8))
        plt.imshow(result_img[:, :, ::-1])
        plt.axis("off")
        plt.show()

        print("Detected:")

        if len(results[0].boxes) == 0:
            print("No road damage detected")
        else:
            for box in results[0].boxes:
                class_id = int(box.cls[0])
                confidence = float(box.conf[0])
                print(f"{names[class_id]} - {confidence:.2f}")

upload.observe(predict_image, names="value")

display(widgets.VBox([
    widgets.HTML("<h3>Road Damage Detection</h3>"),
    widgets.HTML("Drop an image here or click upload:"),
    upload,
    output
]))
